In [3]:
import os
from langchain_openrouter import ChatOpenRouter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n", chunk_size=600, chunk_overlap=100
)

loader = UnstructuredFileLoader("./files/1984.md")

docs = loader.load_and_split(text_splitter=splitter)

llm = ChatOpenRouter(
    temperature=0.1,
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

underlying = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
)
store = LocalFileStore("./.cache/embedding")
embedding = CacheBackedEmbeddings.from_bytes_store(
    underlying,
    store,
    namespace=underlying.model,
)
embedding.model = underlying.model

/var/folders/3y/15zgx_896nsf1dpx6bmzn2180000gn/T/ipykernel_13414/3980248339.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredFileLoader
/var/folders/3y/15zgx_896nsf1dpx6bmzn2180000gn/T/ipykernel_13414/3980248339.py:14: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader("./files/1984.md")
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better resu

In [8]:
from langchain_community.chains import PebbloRetrievalQA
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams


class Qdrant(QdrantVectorStore):
    pass


client = QdrantClient(":memory:")

client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = Qdrant(
    client=client,
    collection_name="demo_collection",
    embedding=embedding,
)
vector_store.add_documents(docs)

chain = PebbloRetrievalQA.from_chain_type(
    retriever=vector_store.as_retriever(),
    llm=llm,
    chain_type="map_rerank",
    app_name="chanoo",
    owner="owner chanoo",
    description="description",
)

chain.run(
    (
        {
            "query": "Where does Winston live?",
            "auth_context": None,
            "semantic_context": None,
        }
    )
)

Unable to reach server http://localhost:8000/v1/app/discover
Unable to reach server http://localhost:8000/v1/prompt/governance
/Users/gimchan-u/Desktop/learn/langchain/env/lib/python3.14/site-packages/langchain_classic/chains/combine_documents/map_rerank.py:190: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  results = self.llm_chain.apply_and_parse(


Question: Where does Winston live?
Helpful Answer: The provided context does not specify where Winston lives (e.g., city, country, district, or building). It only describes his living quarters—a room with a tiny kitchen, a living-room containing a telescreen, and basic furnishings—and mentions that he works at a Ministry (he leaves "the Ministry" and sacrifices his canteen lunch), but his residential location is not stated.
Score: 10Question: Where does Winston live?
Helpful Answer: The provided context does not specify where Winston lives. It describes his living room and mentions a "slummy quarter of the town" where he bought a book, but never names the city, country, or specific location of his residence.
Score: 0Question: Where does Winston live?
Helpful Answer: Winston lives in Victory Mansions in London, which is the chief city of Airstrip One, the third most populous province of Oceania. The Ministry of Truth, his workplace, is about a kilometre away from his residence.
Score: 8

Unable to reach server http://localhost:8000/v1/prompt


Helpful Answer: Winston lives in Victory Mansions in London, which is the chief city of Airstrip One, the third most populous province of Oceania. The Ministry of Truth, his workplace, is about a kilometre away from his residence.